# 01. CV 구종 그룹 분류기 — 학습·검증

`scripts/build_pitch_group_dataset.py`가 생성한 `output/pitch_type_cv/dataset.csv`로
궤적 기반 3그룹(FASTBALL/BREAKING/OFFSPEED) 분류기를 학습하고, 홀드아웃 경기로 검증한다.

**성공 기준: 최빈값 기준선(항상 최다 클래스로 찍기)을 유의미하게 상회하는지.**

랜덤 베이스라인 33%를 쓰면 안 된다. 실측 클래스 분포가 115/103/14로 불균형이라
아무것도 학습하지 않고 항상 FASTBALL만 찍어도 49.6%가 나온다. 33%를 기준으로 삼으면
51.7%짜리 무신호 모델을 성공으로 오판한다 (TS-014에서 실제로 벌어진 일).

In [ ]:
import os
import sys

ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from pitch_type_cv.group_classifier import predict_group, save_classifier, train_classifier
from pitch_type_cv.trajectory_features import FEATURE_COLUMNS

DATASET_PATH = os.path.join(ROOT, "output", "pitch_type_cv", "dataset.csv")
OUT_DIR = os.path.join(ROOT, "output", "pitch_type_cv")

df = pd.read_csv(DATASET_PATH)
print(f"전체 샘플 수: {len(df)}")
df["group"].value_counts()

## 홀드아웃 경기 분리

`game_pk` 중 하나를 통째로 홀드아웃으로 뺀다 (경기 내 데이터 누수를 막기 위해 투구 단위가 아닌
경기 단위로 분리).

In [ ]:
game_pks = df["game_pk"].unique()

# 경기가 1개뿐이면 game_pk 홀드아웃이 불가능하다 (학습셋이 빈다).
# 그 경우 투구 단위 층화 분할로 폴백하되, 같은 경기·같은 투수의 투구가 양쪽에
# 들어가므로 결과는 "누수 허용 상한"이지 일반화 성능이 아니다.
SINGLE_GAME = len(game_pks) == 1

if SINGLE_GAME:
    from sklearn.model_selection import train_test_split

    print(f"경고: 경기가 1개({game_pks[0]})뿐이라 경기 단위 홀드아웃이 불가능합니다.")
    print("      투구 단위 층화 분할로 대체합니다 — 이 수치는 누수 허용 상한입니다.")
    holdout_game_pk = game_pks[0]
    train_df, holdout_df = train_test_split(
        df, test_size=0.3, stratify=df["group"], random_state=42
    )
    train_df = train_df.reset_index(drop=True)
    holdout_df = holdout_df.reset_index(drop=True)
else:
    holdout_game_pk = game_pks[-1]
    print(f"홀드아웃 경기: {holdout_game_pk} (전체 {len(game_pks)}경기 중)")
    train_df = df[df["game_pk"] != holdout_game_pk].reset_index(drop=True)
    holdout_df = df[df["game_pk"] == holdout_game_pk].reset_index(drop=True)

print(f"학습 샘플: {len(train_df)}  홀드아웃 샘플: {len(holdout_df)}")

In [ ]:
model = train_classifier(train_df[FEATURE_COLUMNS + ["group"]], train_df["group"].tolist())

y_true = holdout_df["group"].tolist()
y_pred = [predict_group(model, row.to_dict())[0] for _, row in holdout_df[FEATURE_COLUMNS].iterrows()]

accuracy = accuracy_score(y_true, y_pred)

# 최빈값 기준선: 학습셋의 최다 클래스를 홀드아웃 전체에 찍었을 때의 정확도.
# 아무것도 학습하지 않은 모델이 받는 점수이므로, 이걸 못 넘으면 신호가 없는 것이다.
majority_group = train_df["group"].value_counts().idxmax()
baseline = accuracy_score(y_true, [majority_group] * len(y_true))

print(f"홀드아웃 정확도 : {accuracy:.3f}")
print(f"최빈값 기준선   : {baseline:.3f}  (항상 {majority_group})")
print(f"기준선 대비     : {accuracy - baseline:+.3f}")
print()
print(classification_report(y_true, y_pred, zero_division=0))

## 혼동행렬 & 그룹별 정확도 시각화

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
labels = ["FASTBALL", "BREAKING", "OFFSPEED"]
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues", ax=ax)
ax.set_xlabel("예측")
ax.set_ylabel("실제")
ax.set_title(f"홀드아웃 혼동행렬 (game_pk={holdout_game_pk})")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
per_group_acc = (
    pd.DataFrame({"true": y_true, "pred": y_pred})
    .assign(correct=lambda d: d["true"] == d["pred"])
    .groupby("true")["correct"].mean()
    .reindex(labels)
)
group_counts = pd.Series(y_true).value_counts().reindex(labels).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
per_group_acc.plot(kind="bar", ax=ax, color="#3b82f6")
ax.axhline(baseline, color="red", linestyle="--",
           label=f"최빈값 기준선 {baseline:.1%} (항상 {majority_group})")
ax.set_ylabel("정확도")
ax.set_ylim(0, 1)
ax.set_title(f"그룹별 정확도 — 전체 {accuracy:.1%} vs 기준선 {baseline:.1%}")

# 표본이 적은 그룹의 정확도는 신뢰할 수 없으므로 n을 막대 위에 같이 적는다.
for i, label in enumerate(labels):
    value = per_group_acc.get(label)
    if pd.notna(value):
        ax.text(i, value + 0.02, f"n={group_counts[label]}", ha="center", fontsize=9)

ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "group_accuracy.png"), dpi=150)
plt.show()

In [ ]:
model_path = os.path.join(OUT_DIR, "group_classifier.pkl")
save_classifier(model, model_path)
print(f"모델 저장 완료: {model_path}")

## 해석

**판정 규칙**: `accuracy`가 **최빈값 기준선**을 유의미하게 상회해야 궤적 기반 분류가 통한다는 신호다.
33% 랜덤 베이스라인은 쓰지 않는다 — 클래스 불균형(115/103/14) 때문에 아무것도 학습하지 않아도
49.6%가 나오므로, 33%를 기준으로 삼으면 무신호 모델을 성공으로 오판한다.

**함께 볼 것**:
- **특징 중요도가 평평하면 정확도와 무관하게 신호가 없는 것이다.** 신호가 약할 때는 특정 특징에
  중요도가 몰리고 정확도만 낮다. 7개 특징이 균등하게 0.13~0.16이면 어느 특징에도 정보가 없다는 뜻이다
  (TS-014에서 이 지표가 감지기 오류를 드러낸 유일한 단서였다).
- **표본이 적은 그룹**(OFFSPEED n=14 수준)의 정확도는 참고용으로만 본다. 막대 위 `n` 표시 확인.
- 경기가 1개뿐이면 위 수치는 **누수 허용 상한**이다. 같은 투수의 투구가 학습·홀드아웃 양쪽에 들어간다.
  일반화 성능을 보려면 경기 수를 늘려 `game_pk` 단위 홀드아웃으로 돌려야 한다.

**다음 단계**(별도 스펙): 데이터 규모 확대(경기 수), 다음 구종 예측 파이프라인과의 연결, Streamlit 통합.